In [1]:
import os
os.chdir("..")
import numpy as np
import pandas as pd
np.random.seed(42)

In [2]:
room = pd.read_csv("data/room_occupancy.csv")
room.columns = [c.strip() for c in room.columns]

# ADJUST these column names if Phase 1 Cell 3 showed different names
room["co2_noisy"] = room["CO2"] + np.random.normal(0, 8, len(room))
room["lux_noisy"] = room["Light"] + np.random.normal(0, 6, len(room))

def simulate_posture(n_rows, d0=56, alpha=20, beta=5, lam=0.03):
    posture = np.zeros(n_rows)
    posture[0] = d0
    stress_process = 0.0
    for t in range(1, n_rows):
        fatigue = min(1, t / 3600)
        stress_process += 0.05 * (0 - stress_process) + np.random.normal(0, 0.1)
        stress_process = np.clip(stress_process, 0, 1)
        d_target = d0 - alpha * fatigue - beta * stress_process
        posture[t] = (1 - lam) * posture[t - 1] + lam * d_target + np.random.normal(0, 1.5)
    return posture

room["posture_cm"] = simulate_posture(len(room))
print(room[["co2_noisy", "lux_noisy", "posture_cm"]].describe())

          co2_noisy     lux_noisy    posture_cm
count  20560.000000  20560.000000  20560.000000
mean     690.585150    130.714173     36.695725
std      311.280262    210.577330      7.673651
min      385.056756    -21.930519      9.441490
25%      461.072838     -1.561438     31.555699
50%      564.927219      4.936953     36.326695
75%      805.549275    299.933020     41.207444
max     2077.755749   1707.298810     67.808211


In [3]:
wesad = pd.read_csv("outputs/wesad_raw_features.csv")
print(wesad.columns.tolist())
print(wesad["ground_truth"].value_counts())

['subject', 'hr', 'hrv', 'sdnn', 'pnn50', 'eda', 'wrist_temp', 'wesad_label', 'ground_truth']
ground_truth
NORMAL      1498
STRESSED     641
Name: count, dtype: int64


In [4]:
n = len(wesad)
env_paired = room.iloc[np.arange(n) % len(room)].reset_index(drop=True)
combined = pd.concat([wesad.reset_index(drop=True), env_paired], axis=1)

def apply_rule_engine(row):
    hr, hrv, co2, posture = row["hr"], row["hrv"], row["co2_noisy"], row["posture_cm"]
    if hrv < 25 and hr > 90:
        return "STRESSED"
    if co2 > 1000 or posture < 35:
        return "FATIGUED"
    if hr <= 80 and hrv >= 35 and posture >= 40:
        return "FOCUSED"
    return "NORMAL"

combined["rule_prediction"] = combined.apply(apply_rule_engine, axis=1)
combined["rule_prediction_binary"] = combined["rule_prediction"].apply(
    lambda x: "STRESSED" if x == "STRESSED" else "NORMAL"
)

os.makedirs("outputs", exist_ok=True)
combined.to_csv("outputs/unified_features.csv", index=False)
print("Saved. Rule engine accuracy (binary):",
      (combined["ground_truth"] == combined["rule_prediction_binary"]).mean())

Saved. Rule engine accuracy (binary): 0.7905563347358578
